# 1. Inspect your local federated-learning partition

This notebook runs on your assigned client VM. The mounted dataset is
the partition for your group only; other groups' partitions are not
mounted into this notebook pod.

Run the cells in order. They install the pinned workshop runtime,
identify your group, and inspect the local partition without opening
any source image files.


In [ ]:
%pip install --quiet --user --no-cache-dir --requirement /home/jovyan/digitafrica/workshop/app/requirements.lock


In [ ]:
import os
import sys
from pathlib import Path

group_id = os.environ["GROUP_ID"]
workshop_root = Path(os.environ["DIGITAFRICA_WORKSHOP_ROOT"])
data_path = Path(os.environ["CLIENT_DATA_PATH"])
app_root = workshop_root / "app"

if not group_id.startswith("group_"):
    raise RuntimeError(f"This notebook requires a workshop group login, got {group_id!r}.")
if not data_path.is_file():
    raise FileNotFoundError(f"Local partition is unavailable: {data_path}")
if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

print(f"Workshop group: {group_id}")
print(f"Local partition: {data_path}")
print(f"Workshop mount: {workshop_root}")
print("The workshop mount is read-only; your personal Jupyter workspace remains writable.")


In [ ]:
import pandas as pd

partition = pd.read_csv(data_path)

required_columns = {"image", "label"}
missing_columns = required_columns.difference(partition.columns)
if missing_columns:
    raise ValueError(f"Partition is missing required columns: {sorted(missing_columns)}")

print(f"Samples in this local partition: {len(partition)}")
print("\nClass distribution:")
display(partition["label"].value_counts().sort_index().rename_axis("label").to_frame("samples"))

print("\nFirst rows of local metadata:")
display(partition.head())


In [ ]:
from client.client import load_partition

features, labels = load_partition(
    data_path,
    num_classes=5,
    feature_dim=16,
    seed=42,
)

print(f"Synthetic feature matrix shape: {features.shape}")
print(f"Label vector shape: {labels.shape}")
print(f"Feature dtype: {features.dtype}")
print("Features are deterministic synthetic representations generated from local CSV metadata.")


## Ready for federation

Your group has verified its local partition. Do not start a client
yet: wait for the organiser to confirm that all groups are ready,
then continue with `02_Run_Federated_Client.ipynb`.
